# Various Plots to analyse influence of rain

In [1]:
from tuecycle import list_stations
from tuecycle.plots import list_plots

print(list_stations())  # All station aliases
print(list_plots()) 

from tuecycle.utils.transforms import (
    add_time_features,      # Adds hour, dayofweek, month, year_month, is_weekend
    filter_daytime,         # Filters to hours 6-22
    compute_deviations,     # Adds temp_deviation and bike_deviation columns
    classify_time_category, # Adds time_category (rush hour classification)
    add_season,             # Adds season column (Winter/Transition/Summer)
    prepare_fft_data,       # Prepares data for FFT analysis
)

['freiburg_dreisam', 'freiburg_eschholz', 'freiburg_gueterbahn', 'freiburg_wiwili', 'heidelberg_ernst_walz', 'heidelberg_gaisberg', 'heidelberg_kurfuersten', 'heidelberg_liebermann', 'heidelberg_mannheimer', 'heidelberg_ploeck', 'heidelberg_rohrbacher', 'heidelberg_schlierbacher', 'heidelberg_theodor_heuss', 'heidelberg_ziegelhaeuser', 'heidelberg_berliner', 'heidelberg_eppelheimer', 'heilbronn_neckarufer', 'heilbronn_nord', 'heilbronn_sued', 'karlsruhe_erbprinzen', 'kirchheim_barometer', 'konstanz_herose', 'loerrach_berliner', 'loerrach_friedhof', 'ludwigsburg_alleen', 'ludwigsburg_favorite', 'ludwigsburg_neckarbruecke', 'mannheim_jungbusch', 'mannheim_konrad_adenauer', 'mannheim_kurpfalz', 'mannheim_lindenhof', 'mannheim_renz', 'mannheim_schloss', 'mannheim_schwetzinger', 'mannheim_feudenheimstr_aufwaerts', 'mannheim_feudenheimstr_einwaerts', 'mannheim_luzenbergstr', 'mannheim_b38', 'mannheim_theodor_heuss_in', 'mannheim_theodor_heuss_aus', 'mannheim_fernmeldeturm', 'offenburg_haupt'

1. Set the time window and cities

In [2]:
from tuecycle import DataManager
import pandas as pd
from tuecycle.config.stations import get_stations_by_city

START_DATE = pd.Timestamp(2020, 1, 1)
END_DATE   = pd.Timestamp(2025, 11, 30)

dm = DataManager(
    start_date=(START_DATE.year, START_DATE.month, START_DATE.day),
    end_date=(END_DATE.year, END_DATE.month, END_DATE.day),
)

cities = ["mannheim", "tuebingen", "heidelberg"]



2. Preprocessing

Filter out stations that did not exist at the time of the start date. Preprocess and att mecessary features

In [3]:
from tuecycle.config.stations import get_stations_by_city
from tuecycle.utils.transforms import add_time_features, classify_time_category
import pandas as pd

data_dict = {}       # pro Station
city_data_dict = {}  # pro Stadt, für Plotting etc.

for city in cities:
    stations = get_stations_by_city(city)
    dfs_city = []

    for station in stations:
        try:
            df = dm.get(station.alias)
        except FileNotFoundError:
            print(f"Skipping {station.alias}, bike or weather data missing.")
            continue
        except Exception as e:
            print(f"Skipping {station.alias}, error: {e}")
            continue

        # Prüfen, ob Daten ab START_DATE existieren
        if df['datetime'].max() < START_DATE:
            print(f"Skipping {station.alias}, no data since {START_DATE.date()}")
            continue

        # Zeitfeatures direkt hinzufügen
        df = add_time_features(df)
        df = classify_time_category(df)

        # Pro Station speichern
        data_dict[station.alias] = df

        # Für Stadtaggregation sammeln
        dfs_city.append(df)

    if dfs_city:
        city_data_dict[city] = pd.concat(dfs_city, ignore_index=True)

# Rain share wird später separat berechnet, z.B.:
# rain_share_per_city = compute_rain_share(city_data_dict[city], temp_max=5)


Loading data for Mannheim (Feudenheimstr. stadtauswärts)...
Skipping mannheim_feudenheimstr_aufwaerts, bike or weather data missing.
Loading data for Mannheim (Feudenheimstr. stadteinwärts)...
Skipping mannheim_feudenheimstr_einwaerts, bike or weather data missing.
Loading data for Mannheim (Luzenbergstr.)...
Skipping mannheim_luzenbergstr, bike or weather data missing.
Loading data for Mannheim (B38. RI. AUS)...
Skipping mannheim_b38, bike or weather data missing.
Loading data for Mannheim (Theodor-Heuss-Anlage IN)...
Skipping mannheim_theodor_heuss_in, bike or weather data missing.
Loading data for Mannheim (Theodor-Heuss-Anlage AUS)...
Skipping mannheim_theodor_heuss_aus, bike or weather data missing.
Loading data for Mannheim (Fernmeldeturm)...
Skipping mannheim_fernmeldeturm, bike or weather data missing.
Loading data for Tübingen (Radbrücke Mitte)...
Skipping tuebingen_radbrueckemitte, bike or weather data missing.
Loading data for Tübingen (Radbrücke Ost)...
Skipping tuebingen_r

In [4]:
def compute_rain_share(
    df,
    temp_max,
    temp_min,
    rush_hours=('Morning Rush (7-9)', 'Evening Rush (17-19)')
):
    """
    Returns the share of bike traffic that happens during rain.
    """

    # Analyse-Zeitraum filtern
    df_analysis = df[
        (df['temp'] <= temp_max) &
        (df['temp'] >= temp_min) &
        (df['time_category'].isin(rush_hours))
    ]

    if df_analysis.empty:
        return None

    total_bikes = df_analysis['bike'].sum()
    rain_bikes = df_analysis[df_analysis['rain'] > 0]['bike'].sum()

    if total_bikes == 0:
        return None

    return rain_bikes / total_bikes




5. Define various filter functions

In [5]:
import pandas as pd
import plotly.express as px

def compute_city_rain_shares(city_data_dict, temp_max,temp_min):
    """
    Berechnet den Anteil der Fahrradfahrten bei Regen pro Stadt während der Rush Hour.

    Args:
        city_data_dict: dict {city_name: DataFrame aller Stationen der Stadt}
        temp_max: maximale Temperatur [°C] für die Analyse

    Returns:
        pd.DataFrame mit Spalten ["city", "rain_share"]
    """
    records = []

    for city, city_df in city_data_dict.items():
        share = compute_rain_share(city_df,temp_max, temp_min)
        if share is not None:
            records.append({
                "city": city.capitalize(),
                "rain_share": share
            })

    return pd.DataFrame(records)


def plot_city_rain_shares(df_city_rain, title=None):
    """
    Erstellt einen Boxplot der Regenanteile pro Stadt.

    Args:
        df_city_rain: DataFrame mit Spalten ["city", "rain_share"]
        title: optionaler Plot-Titel

    Returns:
        plotly.graph_objects.Figure
    """
    if title is None:
        title = "Rain share of bike counts per city (Rush Hour)"

    fig = px.box(
        df_city_rain,
        x="city",
        y="rain_share",
        points="all",
        color="city",
        labels={"city": "City", "rain_share": "Rain share of bike counts"},
        title=title
    )

    fig.update_layout(showlegend=False, yaxis_tickformat=".0%")
    return fig


In [6]:
def filter_rain_rush(base_data: dict, temp_max = 50, temp_min = -30) -> dict:
    filtered = {}

    for alias, df in base_data.items():
        df = df.copy()

        # Min-Max-Normalisierung pro Station
        bike_min = df['bike'].min()
        bike_max = df['bike'].max()
        if bike_max > bike_min:
            df['bike_norm'] = (df['bike'] - bike_min) / (bike_max - bike_min)
        else:
            df['bike_norm'] = 0.0  # <- immer setzen

        # Filter für Analyse
        df_f = df[
            (df['temp'] <= temp_max) &
            (df['rain'] == temp_min) &
            (df['time_category'].isin([
                'Morning Rush (7-9)',
                'Evening Rush (17-19)'
            ]))
        ]

        if not df_f.empty:
            filtered[alias] = df_f

    return filtered


6. Plot using the filter functions and defined plot structure

In [7]:
from tuecycle.plots import get_plot

# Plotten
fig = get_plot("bike_vs_rain_rush_hour_city")(
    filter_rain_rush(data_dict, temp_max = 5),
    cities=cities,
    title="Number of cyclists in the rain at ≤5°C (2019-2025) - Rush Hour"
)
fig.show()

# 1️⃣ Berechnung
df_city_rain = compute_city_rain_shares(city_data_dict, temp_max=5, temp_min=-30)


fig = get_plot("city_rain_share_boxplot")(
    df_city_rain,
    title="Rain share ≤5°C, Rush Hour"
)
fig.show()

